In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
# ==========================================================
# PERFORMANCE MEASUREMENT TOOLKIT
# Adds a fair, identical measurement wrapper around every model
# section below (Seasonal Naive, SES, ARIMA, Random Forest, XGBoost,
# LightGBM), so that computation time and memory usage are captured
# with the SAME method for every model:
#
#   - time_sec      : wall-clock seconds for the ENTIRE section
#                      (hyperparameter search + all walk-forward
#                      refits across every category/state series)
#   - peak_mem_MB    : peak memory ALLOCATED BY PYTHON OBJECTS during
#                      that section, measured with tracemalloc
#                      (this captures pandas/numpy/sklearn/statsmodels
#                      allocations; it does NOT capture memory used by
#                      native C/C++ libraries such as LightGBM's or
#                      XGBoost's internal booster buffers - see the
#                      note printed in the summary cell for how to
#                      read this fairly)
#
# model_perf collects these two numbers per model; the final summary
# cell combines them with accuracy (from each *_results_df) and a
# complexity score into one comparison table.
# ==========================================================

import time
import tracemalloc
from contextlib import contextmanager

model_perf = {}

@contextmanager
def measure_block(model_name):
    tracemalloc.start()
    t0 = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - t0
        _current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        model_perf[model_name] = {
            "time_sec": elapsed,
            "peak_mem_MB": peak / (1024 ** 2),
        }
        print(f"\n[measure_block] {model_name}: "
              f"{elapsed:.2f} sec, peak {peak / (1024**2):.2f} MB (tracemalloc)")


In [ ]:
import kagglehub

path = kagglehub.dataset_download("aryayadav0513/m5-forecasting-accuracy")
print("Path:", path)

100%|██████████| 45.8M/45.8M [00:00<00:00, 69.7MB/s]

Extracting files...


Path: /root/.cache/kagglehub/datasets/aryayadav0513/m5-forecasting-accuracy/versions/1


In [ ]:
import os

os.listdir(path)

['m5-forecasting-accuracy']

In [ ]:
import os

dataset_path = os.path.join(path, "m5-forecasting-accuracy")

sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

In [ ]:
print(sales.shape)
print(calendar.shape)
print(prices.shape)

(30490, 1919)
(1969, 14)
(6841121, 4)


In [ ]:
# FILTER STATE DAN CATEGORY
sales = sales[
    (sales['state_id'].isin(['CA','TX','WI'])) &
    (sales['cat_id'].isin(['HOBBIES','HOUSEHOLD','FOODS']))
]

# AMBIL 50% PRODUCT
sales = sales.sample(frac=0.5, random_state=42)

# AMBIL 50% DAYS
day_cols = [c for c in sales.columns if 'd_' in c]
half_days = day_cols[:len(day_cols)//2]

sales = sales[['item_id','dept_id','cat_id','store_id','state_id'] + half_days]

In [ ]:
sales_long = sales.melt(
    id_vars=['item_id','dept_id','cat_id','store_id','state_id'],
    var_name='d',
    value_name='sales'
)

In [ ]:
sales_long = sales_long.merge(
    calendar[['d','wm_yr_wk']],
    on='d',
    how='left'
)

In [ ]:
sales_long = sales_long.merge(
    prices,
    on=['store_id','item_id','wm_yr_wk'],
    how='left'
)

In [ ]:
sales_long['revenue'] = sales_long['sales'] * sales_long['sell_price']

In [ ]:
weekly_avg = sales_long.groupby(
    ['cat_id','state_id','wm_yr_wk']
)['revenue'].sum().reset_index()

In [ ]:
del sales_long
import gc
gc.collect()

37

In [ ]:
hobbies = weekly_avg[weekly_avg['cat_id']=='HOBBIES']
household = weekly_avg[weekly_avg['cat_id']=='HOUSEHOLD']
foods = weekly_avg[weekly_avg['cat_id']=='FOODS']

### Baseline Models: Seasonal Naive & Simple Exponential Smoothing

This section implements two classical **baseline** forecasting methods, used as a lower-bar sanity check against which ARIMA / Random Forest / XGBoost / LightGBM / CNN are compared:

- **Seasonal Naive**: forecasts each week as the value observed `season_length` weeks ago (falls back to the last observed value whenever there isn't enough history yet, e.g. at the very start of the walk-forward window).
- **Simple Exponential Smoothing (SES)**: a weighted average of past observations with exponentially decaying weights (no trend/seasonality component), refit at every step of the walk-forward loop.

**Key steps:**
1. **Walk-forward prediction** — identical protocol to the other sections: derive/fit on history, forecast one week ahead, then append the actual observed test value to history before moving to the next week.
2. **Metrics** — MAE, RMSE, RMSSE, WRMSEE, and MAPE, computed with the same helper functions used elsewhere in the notebook.
3. **Visualization** — real vs. Seasonal Naive vs. SES weekly revenue, one chart pair per category, one line pair per state.


#### Baseline 1 — Seasonal Naive (own block, fair walk-forward)

In [ ]:
with measure_block("Seasonal Naive"):
    import warnings
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    warnings.filterwarnings("ignore")

    # ==========================================================
    # METRICS (identical to the ARIMA / RandomForest / XGBoost / LightGBM /
    # SES sections - same formulas, same masking, same RMSSE denominator)
    # ==========================================================

    def calculate_rmsse(train, test, forecast):
        train = np.array(train).flatten()
        test = np.array(test).flatten()
        forecast = np.array(forecast).flatten()

        denominator = np.mean(np.diff(train) ** 2)
        if denominator < 1e-8:
            return np.nan

        numerator = np.mean((test - forecast) ** 2)
        return np.sqrt(numerator / denominator)


    def calculate_mape(y_true, y_pred):
        y_true = np.array(y_true).flatten()
        y_pred = np.array(y_pred).flatten()

        mask = y_true != 0
        if np.sum(mask) == 0:
            return np.nan

        return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


    # --- Drop the incomplete final boundary week (same as every other section) ---
    last_week = weekly_avg['wm_yr_wk'].max()
    weekly_avg_filtered = weekly_avg[weekly_avg['wm_yr_wk'] != last_week].copy()

    categories = ['HOBBIES', 'HOUSEHOLD', 'FOODS']
    states = ['CA', 'TX', 'WI']
    test_size = 30          # identical to ARIMA / RF / XGB / LightGBM
    lookback = 10            # identical plot-alignment offset used everywhere else

    # ==========================================================
    # HYPERPARAMETER SEARCH (season_length)
    # Every other model in this notebook now searches a hyperparameter and
    # reports what was chosen (ARIMA -> (p, d, q) order, SES -> alpha, RF /
    # XGBoost / LightGBM -> full RandomizedSearchCV param grids). Seasonal
    # Naive previously hardcoded season_length=52 with no search at all,
    # which broke the "fair comparison" premise of this notebook. It is now
    # selected PER SERIES from a small candidate grid by minimizing
    # one-step-ahead MAE on the training history only (test data is never
    # touched during selection), and the chosen value is printed just like
    # every other model's hyperparameter.
    # ==========================================================

    season_length_candidates = [52, 26, 13, 4]  # yearly / half-year / quarterly / monthly weekly-seasonality


    def select_best_season_length(train_series, candidates):
        """Pick the season_length that minimizes one-step-ahead MAE when
        walking through the training series only (no test data is used)."""
        best_len = candidates[0]
        best_mae = float('inf')

        for s in candidates:
            if len(train_series) <= s:
                continue
            errors = [abs(train_series[i] - train_series[i - s]) for i in range(s, len(train_series))]
            if not errors:
                continue
            mae = float(np.mean(errors))
            if mae < best_mae:
                best_mae = mae
                best_len = s

        return best_len

    # ==========================================================
    # FAIR WALK-FORWARD SETTING
    # Every other model in this notebook (ARIMA, RF, XGBoost, LightGBM) is
    # evaluated by walking forward one week at a time over an EXPANDING
    # history and re-deriving its forecast from that history at every step
    # (ARIMA refits, RF/XGB/LightGBM refit). Seasonal Naive has no
    # parameters to "refit", but to stay on the same footing we still walk
    # forward one step at a time and re-read the forecast from the
    # expanding history at every step, instead of forecasting the whole
    # test block in one shot from the original train set.
    # ==========================================================

    # Pre-calculate absolute revenue weights for WRMSSE scaling
    # (recomputed locally so this cell is fully self-contained and uses
    # exactly the same weighting as every other model section)
    revenue_weights = {}
    grand_total_revenue = 0

    for cat in categories:
        for state in states:
            df_subset = weekly_avg_filtered[
                (weekly_avg_filtered['cat_id'] == cat) &
                (weekly_avg_filtered['state_id'] == state)
            ]
            if not df_subset.empty:
                total_rev = df_subset['revenue'].sum()
                revenue_weights[(cat, state)] = total_rev
                grand_total_revenue += total_rev

    # Plot aesthetics
    colors_real = {'CA': 'tab:blue', 'TX': 'tab:green', 'WI': 'tab:purple'}
    colors_snaive = {'CA': 'tab:orange', 'TX': 'tab:red', 'WI': 'tab:brown'}

    snaive_results = []      # per-series metrics for this model
    all_wrmsee_snaive = []   # cumulative WRMSSE tracker, same pattern as RF/XGB/LightGBM cells

    for cat in categories:
        plt.figure(figsize=(12, 6))
        print("=" * 60)
        print(f" CATEGORY: {cat} ")
        print("=" * 60)

        for state in states:
            df_subset = weekly_avg_filtered[
                (weekly_avg_filtered['cat_id'] == cat) &
                (weekly_avg_filtered['state_id'] == state)
            ].sort_values('wm_yr_wk')

            series = df_subset['revenue'].values
            if len(series) < test_size + 1:
                continue

            train = series[:-test_size]
            test = series[-test_size:]

            weight = revenue_weights.get((cat, state), 0) / grand_total_revenue

            # ------------------------------------------------------------
            # HYPERPARAMETER SELECTION - season_length, searched on train only
            # ------------------------------------------------------------
            season_length = select_best_season_length(train, season_length_candidates)
            print(f"{cat}-{state} | Selected hyperparameter season_length: {season_length}")

            # ------------------------------------------------------------
            # SEASONAL NAIVE - walk-forward over an expanding history,
            # exactly like the ARIMA(history, order=best_order) loop:
            # at every step we look season_length steps back into the
            # CURRENT history (train + observed test so far), then append
            # the true observed value (never the prediction) before moving on.
            # ------------------------------------------------------------
            history = list(train)
            snaive_preds = []
            for t in range(len(test)):
                if len(history) >= season_length:
                    yhat = history[-season_length]
                else:
                    yhat = history[-1]   # not enough history yet -> plain naive fallback
                snaive_preds.append(yhat)
                history.append(test[t])   # walk-forward: append the actual, not the forecast
            snaive_preds = np.array(snaive_preds)

            # In-sample fitted values for the plot only (same fallback logic,
            # plays no part in the metrics above - same role as ARIMA's
            # `initial_model.fittedvalues` / RF's `initial_model.predict(X_train)`)
            snaive_in_sample = np.array([
                train[i - season_length] if i >= season_length else (train[i - 1] if i >= 1 else train[0])
                for i in range(len(train))
            ])
            snaive_full_pred = np.concatenate([snaive_in_sample, snaive_preds])

            # ------------------------------------------------------------
            # METRICS
            # ------------------------------------------------------------
            mae = mean_absolute_error(test, snaive_preds)
            rmse = np.sqrt(mean_squared_error(test, snaive_preds))
            mape_val = calculate_mape(test, snaive_preds)
            rmsse = calculate_rmsse(train, test, snaive_preds)
            wrmsee = weight * rmsse
            all_wrmsee_snaive.append(wrmsee)

            snaive_results.append({
                "category": cat, "state": state, "model": "Seasonal Naive",
                "MAE": mae, "RMSE": rmse, "RMSSE": rmsse,
                "WRMSSE": wrmsee, "MAPE": mape_val, "season_length": season_length
            })

            print(f"{cat} - {state} (Seasonal Naive)")
            print(f"MAE   : {mae:.4f}")
            print(f"RMSE  : {rmse:.4f}")
            print(f"WRMSSE: {wrmsee:.4f} (Weight: {weight:.4f}, Cumulative: {np.nansum(all_wrmsee_snaive):.4f})")
            print(f"MAPE  : {mape_val:.2f}%")
            print("-" * 60)

            # ------------------------------------------------------------
            # PLOT - truncate lookback elements to align with the RNN's data shape
            # ------------------------------------------------------------
            plot_real = series[lookback:]
            plot_pred = snaive_full_pred[lookback:]

            plt.plot(plot_real, label=f'{state} Real', color=colors_real[state], linestyle='-')
            plt.plot(plot_pred, label=f'{state} Seasonal Naive', color=colors_snaive[state], linestyle='--')

        plt.title(f"{cat} Revenue \u2014 Seasonal Naive (fair walk-forward)", fontsize=13)
        plt.xlabel("Week", fontsize=11)
        plt.ylabel("Average Revenue", fontsize=11)

        ax = plt.gca()
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f'{int(x/1000)}k' if x >= 1000 else f'{int(x)}'))
        plt.legend(loc='upper left', frameon=True, fontsize=8)
        plt.grid(True, linestyle=':', alpha=0.5)
        plt.tight_layout()
        plt.show()

    snaive_results_df = pd.DataFrame(snaive_results)

    print("\n" + "#" * 60)
    print(f"Overall WRMSSE (Seasonal Naive): {np.nansum(all_wrmsee_snaive):.4f}")
    print("#" * 60)

    snaive_results_df


#### Baseline 2 — Simple Exponential Smoothing (own block, fair walk-forward)

In [ ]:
with measure_block("SES"):
    import warnings
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    from statsmodels.tsa.holtwinters import SimpleExpSmoothing
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    warnings.filterwarnings("ignore")

    # ==========================================================
    # METRICS (identical to the ARIMA / RandomForest / XGBoost / LightGBM /
    # Seasonal Naive sections - same formulas, same masking, same RMSSE denominator)
    # ==========================================================

    def calculate_rmsse(train, test, forecast):
        train = np.array(train).flatten()
        test = np.array(test).flatten()
        forecast = np.array(forecast).flatten()

        denominator = np.mean(np.diff(train) ** 2)
        if denominator < 1e-8:
            return np.nan

        numerator = np.mean((test - forecast) ** 2)
        return np.sqrt(numerator / denominator)


    def calculate_mape(y_true, y_pred):
        y_true = np.array(y_true).flatten()
        y_pred = np.array(y_pred).flatten()

        mask = y_true != 0
        if np.sum(mask) == 0:
            return np.nan

        return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


    # --- Drop the incomplete final boundary week (same as every other section) ---
    last_week = weekly_avg['wm_yr_wk'].max()
    weekly_avg_filtered = weekly_avg[weekly_avg['wm_yr_wk'] != last_week].copy()

    categories = ['HOBBIES', 'HOUSEHOLD', 'FOODS']
    states = ['CA', 'TX', 'WI']
    test_size = 30    # identical to ARIMA / RF / XGB / LightGBM / Seasonal Naive
    lookback = 10      # identical plot-alignment offset used everywhere else

    # ==========================================================
    # FAIR WALK-FORWARD SETTING
    # Just like ARIMA(history, order=best_order).fit() is re-run at every
    # step, SES is re-fit from scratch on the expanding history at every
    # step (SimpleExpSmoothing(history).fit(optimized=True)), so its
    # smoothing level (alpha) is always re-estimated from all data
    # available up to that point - no information from the future leaks in,
    # and no single stale fit is reused for the whole test horizon.
    # ==========================================================

    # Pre-calculate absolute revenue weights for WRMSSE scaling
    # (recomputed locally so this cell is fully self-contained and uses
    # exactly the same weighting as every other model section)
    revenue_weights = {}
    grand_total_revenue = 0

    for cat in categories:
        for state in states:
            df_subset = weekly_avg_filtered[
                (weekly_avg_filtered['cat_id'] == cat) &
                (weekly_avg_filtered['state_id'] == state)
            ]
            if not df_subset.empty:
                total_rev = df_subset['revenue'].sum()
                revenue_weights[(cat, state)] = total_rev
                grand_total_revenue += total_rev

    # Plot aesthetics
    colors_real = {'CA': 'tab:blue', 'TX': 'tab:green', 'WI': 'tab:purple'}
    colors_ses = {'CA': 'gold', 'TX': 'deeppink', 'WI': 'slategray'}

    ses_results = []      # per-series metrics for this model
    all_wrmsee_ses = []   # cumulative WRMSSE tracker, same pattern as RF/XGB/LightGBM cells

    for cat in categories:
        plt.figure(figsize=(12, 6))
        print("=" * 60)
        print(f" CATEGORY: {cat} ")
        print("=" * 60)

        for state in states:
            df_subset = weekly_avg_filtered[
                (weekly_avg_filtered['cat_id'] == cat) &
                (weekly_avg_filtered['state_id'] == state)
            ].sort_values('wm_yr_wk')

            series = df_subset['revenue'].values
            if len(series) < test_size + 1:
                continue

            train = series[:-test_size]
            test = series[-test_size:]

            weight = revenue_weights.get((cat, state), 0) / grand_total_revenue

            # ------------------------------------------------------------
            # SIMPLE EXPONENTIAL SMOOTHING - walk-forward, refit every step
            # on the expanding history, exactly mirroring the ARIMA loop.
            # ------------------------------------------------------------
            history = list(train)
            ses_preds = []
            ses_alphas = []   # hyperparameter (smoothing_level / alpha) chosen at each refit
            for t in range(len(test)):
                ses_model = SimpleExpSmoothing(history, initialization_method="estimated").fit(optimized=True)
                alpha = ses_model.params.get("smoothing_level", np.nan)
                ses_alphas.append(alpha)
                yhat = ses_model.forecast(1)[0]
                ses_preds.append(yhat)
                history.append(test[t])   # walk-forward: append the actual, not the forecast
            ses_preds = np.array(ses_preds)
            avg_alpha = float(np.nanmean(ses_alphas))

            # In-sample fitted values for the plot only (single fit on train,
            # plays no part in the metrics above - same role as ARIMA's
            # `initial_model.fittedvalues` / RF's `initial_model.predict(X_train)`)
            ses_initial = SimpleExpSmoothing(train, initialization_method="estimated").fit(optimized=True)
            ses_in_sample = ses_initial.fittedvalues
            ses_full_pred = np.concatenate([ses_in_sample, ses_preds])

            # ------------------------------------------------------------
            # METRICS
            # ------------------------------------------------------------
            mae = mean_absolute_error(test, ses_preds)
            rmse = np.sqrt(mean_squared_error(test, ses_preds))
            mape_val = calculate_mape(test, ses_preds)
            rmsse = calculate_rmsse(train, test, ses_preds)
            wrmsee = weight * rmsse
            all_wrmsee_ses.append(wrmsee)

            ses_results.append({
                "category": cat, "state": state, "model": "SES",
                "MAE": mae, "RMSE": rmse, "RMSSE": rmsse,
                "WRMSSE": wrmsee, "MAPE": mape_val, "alpha": avg_alpha
            })

            print(f"{cat} - {state} (SES)")
            print(f"Hyperparameter alpha (smoothing_level) - mean over walk-forward refits: {avg_alpha:.4f} "
                  f"(last step: {ses_alphas[-1]:.4f})")
            print(f"MAE   : {mae:.4f}")
            print(f"RMSE  : {rmse:.4f}")
            print(f"WRMSSE: {wrmsee:.4f} (Weight: {weight:.4f}, Cumulative: {np.nansum(all_wrmsee_ses):.4f})")
            print(f"MAPE  : {mape_val:.2f}%")
            print("-" * 60)

            # ------------------------------------------------------------
            # PLOT - truncate lookback elements to align with the RNN's data shape
            # ------------------------------------------------------------
            plot_real = series[lookback:]
            plot_pred = ses_full_pred[lookback:]

            plt.plot(plot_real, label=f'{state} Real', color=colors_real[state], linestyle='-')
            plt.plot(plot_pred, label=f'{state} SES', color=colors_ses[state], linestyle='--')

        plt.title(f"{cat} Revenue \u2014 Simple Exponential Smoothing (fair walk-forward)", fontsize=13)
        plt.xlabel("Week", fontsize=11)
        plt.ylabel("Average Revenue", fontsize=11)

        ax = plt.gca()
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f'{int(x/1000)}k' if x >= 1000 else f'{int(x)}'))
        plt.legend(loc='upper left', frameon=True, fontsize=8)
        plt.grid(True, linestyle=':', alpha=0.5)
        plt.tight_layout()
        plt.show()

    ses_results_df = pd.DataFrame(ses_results)

    print("\n" + "#" * 60)
    print(f"Overall WRMSSE (SES): {np.nansum(all_wrmsee_ses):.4f}")
    print("#" * 60)

    ses_results_df


In [ ]:
# Quick summary: average WRMSSE / MAPE per baseline model, across all category-state series
baseline_results_df = pd.concat([snaive_results_df, ses_results_df], ignore_index=True)
baseline_summary = baseline_results_df.groupby("model")[["MAE", "RMSE", "RMSSE", "WRMSSE", "MAPE"]].mean()
baseline_summary


,MAE,RMSE,RMSSE,WRMSSE,MAPE
model,,,,,
SES,2351.513004,2876.858585,0.920128,0.098105,5.651469
Seasonal Naive,2201.439593,2859.230731,1.152660,0.109782,6.026710


### ARIMA Revenue Forecast

This section implements an **ARIMA** (AutoRegressive Integrated Moving Average) model as a classical statistical baseline for each category/state weekly revenue series.

**Key steps:**
1. **Order selection** — for each series, search `(p, d, q)` combinations (capped at order 2) and pick the one that minimizes AIC on the training data, keeping the model parsimonious.
2. **Walk-forward prediction** — fit on history, forecast one week ahead, then append the *actual* observed test value to history before refitting for the next week (same strategy used by the other model sections).
3. **Metrics** — MAE, RMSE, RMSSE, WRMSEE, and MAPE, computed with the same helper functions used elsewhere in the notebook.
4. **Visualization** — real vs. ARIMA-forecast weekly revenue per category and state.

In [ ]:
with measure_block("ARIMA"):
    import warnings
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    from statsmodels.tsa.arima.model import ARIMA
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    # Suppress convergence warnings from non-matching grid search orders
    warnings.filterwarnings("ignore")

    # 1. Helper metrics matching M5 competition standards
    # (identical formulas/masking to every other section in the notebook -
    #  the original local copies here diverged: no zero-masking in MAPE, and
    #  a degenerate denominator returned 0 instead of NaN. Fixed to match.)
    def calculate_rmsse(train, test, forecast):
        train = np.array(train).flatten()
        test = np.array(test).flatten()
        forecast = np.array(forecast).flatten()

        denominator = np.mean(np.diff(train) ** 2)
        if denominator < 1e-8:
            return np.nan
        numerator = np.mean((test - forecast) ** 2)
        return np.sqrt(numerator / denominator)

    def calculate_mape(y_true, y_pred):
        y_true = np.array(y_true).flatten()
        y_pred = np.array(y_pred).flatten()

        mask = y_true != 0
        if np.sum(mask) == 0:
            return np.nan
        return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    # 2. Anti-Overfitting Hyperparameter Optimization Engine
    def find_best_arima_order(train_series):
        """
        Finds the optimal (p, d, q) order using AIC to minimize overfitting risks.
        We cap orders at 2 to keep the model parsimonious and stable.
        """
        best_aic = float('inf')
        best_order = (1, 1, 1)  # Robust default fallback

        for p in [0, 1, 2]:
            for d in [0, 1]:
                for q in [0, 1, 2]:
                    try:
                        model = ARIMA(train_series, order=(p, d, q))
                        results = model.fit()
                        if results.aic < best_aic:
                            best_aic = results.aic
                            best_order = (p, d, q)
                    except:
                        continue
        return best_order

    # --- Drop the incomplete final boundary week ---
    last_week = weekly_avg['wm_yr_wk'].max()
    weekly_avg_filtered = weekly_avg[weekly_avg['wm_yr_wk'] != last_week].copy()

    categories = ['HOBBIES', 'HOUSEHOLD', 'FOODS']
    states = ['CA', 'TX', 'WI']
    test_size = 30
    lookback = 10  # Aligns index with your RNN lookback window

    # 3. Pre-calculate absolute revenue weights for WRMSEE scaling
    revenue_weights = {}
    grand_total_revenue = 0

    for cat in categories:
        for state in states:
            df_subset = weekly_avg_filtered[
                (weekly_avg_filtered['cat_id'] == cat) &
                (weekly_avg_filtered['state_id'] == state)
            ]
            if not df_subset.empty:
                total_rev = df_subset['revenue'].sum()
                revenue_weights[(cat, state)] = total_rev
                grand_total_revenue += total_rev

    # Plot aesthetics
    colors_real = {'CA': 'tab:blue', 'TX': 'tab:green', 'WI': 'tab:purple'}
    colors_arima = {'CA': 'tab:orange', 'TX': 'tab:red', 'WI': 'tab:brown'}

    arima_results = []  # per-series metrics, mirrors snaive_results / ses_results

    # 4. Execution and Visual Alignment Loop
    for cat in categories:
        plt.figure(figsize=(12, 6))
        print("=" * 60)
        print(f" CATEGORY: {cat} ")
        print("=" * 60)

        for state in states:
            df_subset = weekly_avg_filtered[
                (weekly_avg_filtered['cat_id'] == cat) &
                (weekly_avg_filtered['state_id'] == state)
            ].sort_values('wm_yr_wk')

            series = df_subset['revenue'].values
            if len(series) < test_size + 1:
                # not enough history for a full test window - same guard used
                # by every other model section (RF/XGB/LightGBM/Seasonal Naive/SES);
                # previously this only checked for an empty series, which let a
                # too-short series through and produced an empty/garbage train split.
                continue

            train = series[:-test_size]
            test = series[-test_size:]

            weight = revenue_weights.get((cat, state), 0) / grand_total_revenue

            # Find the optimal order on training data to prevent overfitting
            best_order = find_best_arima_order(train)

            # Walk-forward tracking using the optimal parsimonious order
            history = list(train)
            rolling_predictions = []

            for t in range(len(test)):
                model = ARIMA(history, order=best_order)
                model_fitted = model.fit()
                yhat = model_fitted.forecast(steps=1)[0]
                rolling_predictions.append(yhat)
                history.append(test[t])

            rolling_predictions = np.array(rolling_predictions)

            # Generate full baseline timeline matching the series shape
            initial_model = ARIMA(train, order=best_order).fit()
            in_sample = initial_model.fittedvalues
            full_pred = np.concatenate([in_sample, rolling_predictions])

            # Metrics computation
            mae = mean_absolute_error(test, rolling_predictions)
            rmse = np.sqrt(mean_squared_error(test, rolling_predictions))
            mape_val = calculate_mape(test, rolling_predictions)
            rmsse = calculate_rmsse(train, test, rolling_predictions)
            wrmsee = weight * rmsse

            arima_results.append({
                "category": cat, "state": state, "model": "ARIMA",
                "MAE": mae, "RMSE": rmse, "RMSSE": rmsse,
                "WRMSSE": wrmsee, "MAPE": mape_val, "order": best_order
            })

            # Printed metric summary
            print(f"{cat} - {state} (Selected Order: ARIMA{best_order})")
            print(f"MAE   : {mae:.4f}")
            print(f"RMSE  : {rmse:.4f}")
            print(f"WRMSSE: {wrmsee:.4f} (Weight: {weight:.4f})")
            print(f"MAPE  : {mape_val:.2f}%")
            print("-" * 60)

            # Truncate lookback elements to align perfectly with the RNN's data shape
            plot_real = series[lookback:]
            plot_pred = full_pred[lookback:]

            plt.plot(plot_real, label=f'{state} Real', color=colors_real[state], linestyle='-')
            plt.plot(plot_pred, label=f'{state} ARIMA', color=colors_arima[state], linestyle='--')

        # Chart polish
        plt.title(f"{cat} Revenue Forecast (AIC-Optimized ARIMA)", fontsize=14, pad=10)
        plt.xlabel("Week", fontsize=11)
        plt.ylabel("Average Revenue", fontsize=11)

        ax = plt.gca()
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f'{int(x/1000)}k' if x >= 1000 else f'{int(x)}'))

        plt.legend(loc='upper left', frameon=True)
        plt.grid(True, linestyle=':', alpha=0.5)
        plt.tight_layout()
        plt.show()

    arima_results_df = pd.DataFrame(arima_results)

    print("\n" + "#" * 60)
    print(f"Overall WRMSSE (ARIMA): {arima_results_df['WRMSSE'].sum():.4f}")
    print("#" * 60)

    arima_results_df


### Random Forest Revenue Forecast

This section implements a **Random Forest** regressor as a tree-ensemble alternative to ARIMA/LightGBM.

Like the LightGBM section, each week is converted into a tabular row containing:

- `lag_1 ... lag_10`: revenue from the previous 10 weeks
- `roll_mean_w` / `roll_std_w`: rolling mean/std over 3, 5, and 10-week windows
- `week_idx`: a simple trend index

**Key steps:**
1. **Feature engineering** — convert each category/state revenue series into a lagged tabular dataset.
2. **Hyperparameter search** — a global `RandomizedSearchCV` over Random Forest parameters (`n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`).
3. **Per-series fit** — refit a Random Forest with the best hyperparameters for each category/state pair.
4. **Walk-forward prediction** — predict one week at a time, then append the *actual* observed test value to history before building next week's features.
5. **Metrics** — MAE, RMSE, RMSSE, WRMSEE, and MAPE, computed with the same helper functions used elsewhere in the notebook.
6. **Visualization** — real vs. Random Forest-forecast weekly revenue per category and state.

In [ ]:
with measure_block("Random Forest"):
    import warnings
    import gc
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

    from sklearn.metrics import mean_absolute_error, mean_squared_error
    from sklearn.model_selection import RandomizedSearchCV
    from sklearn.ensemble import RandomForestRegressor

    warnings.filterwarnings("ignore")

    # ==========================================================
    # METRICS (identical to the ARIMA / LightGBM sections)
    # ==========================================================

    def calculate_rmsse(train, test, forecast):
        train = np.array(train).flatten()
        test = np.array(test).flatten()
        forecast = np.array(forecast).flatten()

        denominator = np.mean(np.diff(train) ** 2)
        if denominator < 1e-8:
            return np.nan

        numerator = np.mean((test - forecast) ** 2)
        return np.sqrt(numerator / denominator)


    def calculate_mape(y_true, y_pred):
        y_true = np.array(y_true).flatten()
        y_pred = np.array(y_pred).flatten()

        mask = y_true != 0
        if np.sum(mask) == 0:
            return np.nan

        return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


    # ==========================================================
    # FEATURE ENGINEERING
    # ==========================================================

    time_step = 10            # same lookback window used elsewhere in the notebook
    n_lags = time_step        # number of lag features
    roll_windows = [3, 5, 10]  # rolling mean/std windows (must be <= n_lags)


    def build_feature_frame(series: np.ndarray, n_lags: int, roll_windows) -> pd.DataFrame:
        """
        Turns a 1D revenue series into a tabular feature set:
          lag_1 ... lag_n         -> previous n weeks
          roll_mean_w / roll_std_w -> rolling stats over lag_1..lag_w
          week_idx                 -> simple trend index
        Target column 'y' is the current week's revenue.
        """
        df = pd.DataFrame({"y": series})

        for lag in range(1, n_lags + 1):
            df[f"lag_{lag}"] = df["y"].shift(lag)

        for w in roll_windows:
            lag_cols = [f"lag_{i}" for i in range(1, w + 1)]
            df[f"roll_mean_{w}"] = df[lag_cols].mean(axis=1)
            df[f"roll_std_{w}"] = df[lag_cols].std(axis=1)

        df["week_idx"] = np.arange(len(df))

        df = df.dropna().reset_index(drop=True)
        feature_cols = [c for c in df.columns if c != "y"]
        return df, feature_cols


    # ==========================================================
    # HYPERPARAMETER SEARCH SPACE
    # (search is now run separately for EACH cat/state series, so every
    #  series gets its own tuned model instead of one global model -
    #  this puts RF on equal footing with the per-series ARIMA orders)
    # ==========================================================

    param_dist = {
        "n_estimators": [200, 400, 600, 800],
        "max_depth": [None, 4, 6, 8, 12],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 1.0],
    }

    N_ITER = 15   # smaller than before since this now runs once PER series (9x total)
    CV_FOLDS = 3

    # ==========================================================
    # FAIR WALK-FORWARD SETTING
    # ARIMA refits ARIMA(history, order=best_order) at EVERY test step, so it
    # always trains on all data available up to that point. To make RF an
    # apples-to-apples comparison, we now also refit RF at every walk-forward
    # step on the expanding history, using the best_params found once per
    # series (the tree analogue of ARIMA's fixed best_order). This is much
    # more expensive computationally (test_size refits per series instead of
    # one), but it removes RF's "static model" advantage/disadvantage.
    # ==========================================================

    REFIT_EACH_STEP = True

    # ==========================================================
    # PARAMETERS FOR PER-SERIES PLOTS
    # ==========================================================

    colors_real = {"CA": "tab:blue", "TX": "tab:green", "WI": "tab:purple"}
    colors_rf = {"CA": "tab:orange", "TX": "tab:red", "WI": "tab:brown"}

    all_wrmsee_rf = []
    best_params_per_series_rf = {}
    rf_results = []  # per-series metrics, mirrors snaive_results / ses_results / arima_results

    # ==========================================================
    # FORECASTING LOOP (tuning now happens INSIDE this loop, per series)
    # ==========================================================

    for cat in categories:
        plt.figure(figsize=(12, 6))

        print("=" * 60)
        print(f"CATEGORY : {cat}")
        print("=" * 60)

        for state in states:
            df_subset = weekly_avg_filtered[
                (weekly_avg_filtered["cat_id"] == cat) &
                (weekly_avg_filtered["state_id"] == state)
            ].sort_values("wm_yr_wk")

            series = df_subset["revenue"].values
            if len(series) < n_lags + test_size + 1:
                continue

            # ==========================================
            # TRAIN / TEST SPLIT
            # ==========================================

            train_data = series[:-test_size]
            test_data = series[-test_size:]

            train_feat_df, feature_cols = build_feature_frame(train_data, n_lags, roll_windows)
            X_train = train_feat_df[feature_cols]
            y_train = train_feat_df["y"]

            # ==========================================
            # PER-SERIES HYPERPARAMETER SEARCH
            # (done ONCE on the training data only, exactly like ARIMA's
            #  best_order search - the walk-forward refits below reuse
            #  these best_params, they don't re-search every step)
            # ==========================================

            n_splits = min(CV_FOLDS, max(2, len(X_train) // 5))

            base_model = RandomForestRegressor(random_state=42, n_jobs=-1)
            search = RandomizedSearchCV(
                base_model,
                param_distributions=param_dist,
                n_iter=N_ITER,
                scoring="neg_mean_squared_error",
                cv=n_splits,
                random_state=42,
                n_jobs=-1,
                verbose=0,
            )
            search.fit(X_train, y_train)

            best_params = search.best_params_
            best_params_per_series_rf[(cat, state)] = best_params

            print(f"{cat}-{state} | Best RF params: {best_params}")

            # ==========================================
            # WALK-FORWARD PREDICTION (model refit at EVERY step,
            # exactly like the ARIMA(history, order=best_order).fit()
            # call inside the ARIMA walk-forward loop)
            # ==========================================

            history = list(train_data)
            predictions = []

            for i in range(len(test_data)):
                hist_arr = np.array(history)

                # REFIT_EACH_STEP is always True in this notebook: the model is
                # refit from scratch on the expanding history at every walk-forward
                # step, exactly like ARIMA(history, order=best_order).fit().
                hist_feat_df, _ = build_feature_frame(hist_arr, n_lags, roll_windows)
                X_hist = hist_feat_df[feature_cols]
                y_hist = hist_feat_df["y"]

                step_model = RandomForestRegressor(
                    **best_params, random_state=42, n_jobs=-1
                )
                step_model.fit(X_hist, y_hist)

                feat_row = {}

                for lag in range(1, n_lags + 1):
                    feat_row[f"lag_{lag}"] = hist_arr[-lag]

                for w in roll_windows:
                    lag_vals = hist_arr[-w:]
                    feat_row[f"roll_mean_{w}"] = np.mean(lag_vals)
                    feat_row[f"roll_std_{w}"] = np.std(lag_vals, ddof=1)  # ddof=1 to match pandas .std() used in build_feature_frame

                feat_row["week_idx"] = len(hist_arr)

                X_input = pd.DataFrame([feat_row])[feature_cols]
                pred = step_model.predict(X_input)[0]
                predictions.append(pred)

                # walk-forward: append the actual observed value, not the prediction
                history.append(test_data[i])

            predictions = np.array(predictions)
            actual = test_data

            # ==========================================
            # METRICS
            # ==========================================

            mae = mean_absolute_error(actual, predictions)
            rmse = np.sqrt(mean_squared_error(actual, predictions))
            mape = calculate_mape(actual, predictions)
            rmsse = calculate_rmsse(train_data, actual, predictions)

            weight = revenue_weights[(cat, state)] / grand_total_revenue
            wrmsee = weight * rmsse
            all_wrmsee_rf.append(wrmsee)


            rf_results.append({
                "category": cat, "state": state, "model": "Random Forest",
                "MAE": mae, "RMSE": rmse, "RMSSE": rmsse,
                "WRMSSE": wrmsee, "MAPE": mape, "best_params": best_params
            })
            print(f"MAE    : {mae:.4f}")
            print(f"RMSE   : {rmse:.4f}")
            print(f"RMSSE  : {rmsse:.4f}")
            print(f"WRMSEE : {wrmsee:.4f} (Cumulative: {np.nansum(all_wrmsee_rf):.4f})")
            print(f"MAPE   : {mape:.2f}%")
            print("-" * 50)

            # ==========================================
            # TRAIN FIT FOR PLOT
            # (a single model fit on train_data only, used purely to draw the
            #  in-sample line - same role as ARIMA's `initial_model` used for
            #  `in_sample = initial_model.fittedvalues`. It plays no part in
            #  the walk-forward predictions/metrics above.)
            # ==========================================

            initial_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
            initial_model.fit(X_train, y_train)
            train_pred = initial_model.predict(X_train)

            full_pred = np.empty(len(series))
            full_pred[:] = np.nan
            full_pred[n_lags: n_lags + len(train_pred)] = train_pred
            full_pred[len(series) - len(predictions):] = predictions

            plt.plot(series[n_lags:], label=f"{state} Real", color=colors_real[state])
            plt.plot(full_pred[n_lags:], "--", label=f"{state} Random Forest", color=colors_rf[state])

        plt.title(f"{cat} Revenue Forecast (Random Forest, per-series tuning, refit-per-step)")
        plt.xlabel("Week")
        plt.ylabel("Revenue")

        ax = plt.gca()
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, pos: f"{int(x/1000)}k" if x >= 1000 else f"{int(x)}")
        )

        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        gc.collect()

    # ==========================================================
    # OVERALL WRMSEE
    # ==========================================================

    print("\n" + "#" * 60)
    print(f"Overall WRMSEE : {np.nansum(all_wrmsee_rf):.4f}")
    print("#" * 60)

    rf_results_df = pd.DataFrame(rf_results)

    print("\n" + "#" * 60)
    print(f"Overall WRMSEE (Random Forest): {rf_results_df['WRMSSE'].sum():.4f}")
    print("#" * 60)

    rf_results_df


### XGBoost Revenue Forecast

This section implements an **XGBoost** gradient-boosted-tree model, using the same tabular feature representation as the LightGBM and Random Forest sections.

Each week is converted into a row containing:

- `lag_1 ... lag_10`: revenue from the previous 10 weeks
- `roll_mean_w` / `roll_std_w`: rolling mean/std over 3, 5, and 10-week windows
- `week_idx`: a simple trend index

**Key steps:**
1. **Feature engineering** — convert each category/state revenue series into a lagged tabular dataset.
2. **Hyperparameter search** — a global `RandomizedSearchCV` over XGBoost parameters (`max_depth`, `learning_rate`, `n_estimators`, `subsample`, `colsample_bytree`, etc.).
3. **Per-series fit** — refit XGBoost with the best hyperparameters for each category/state pair.
4. **Walk-forward prediction** — predict one week at a time, then append the *actual* observed test value to history before building next week's features.
5. **Metrics** — MAE, RMSE, RMSSE, WRMSEE, and MAPE, computed with the same helper functions used elsewhere in the notebook.
6. **Visualization** — real vs. XGBoost-forecast weekly revenue per category and state.

Install once if needed: `!pip install xgboost`

In [ ]:
!pip install xgboost

In [ ]:
with measure_block("XGBoost"):
    import warnings
    import gc
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

    from sklearn.metrics import mean_absolute_error, mean_squared_error
    from sklearn.model_selection import RandomizedSearchCV
    import xgboost as xgb

    warnings.filterwarnings("ignore")

    # ==========================================================
    # METRICS (identical to the ARIMA / LightGBM / Random Forest sections)
    # ==========================================================

    def calculate_rmsse(train, test, forecast):
        train = np.array(train).flatten()
        test = np.array(test).flatten()
        forecast = np.array(forecast).flatten()

        denominator = np.mean(np.diff(train) ** 2)
        if denominator < 1e-8:
            return np.nan

        numerator = np.mean((test - forecast) ** 2)
        return np.sqrt(numerator / denominator)


    def calculate_mape(y_true, y_pred):
        y_true = np.array(y_true).flatten()
        y_pred = np.array(y_pred).flatten()

        mask = y_true != 0
        if np.sum(mask) == 0:
            return np.nan

        return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


    # ==========================================================
    # FEATURE ENGINEERING
    # ==========================================================

    time_step = 10            # same lookback window used elsewhere in the notebook
    n_lags = time_step        # number of lag features
    roll_windows = [3, 5, 10]  # rolling mean/std windows (must be <= n_lags)


    def build_feature_frame(series: np.ndarray, n_lags: int, roll_windows) -> pd.DataFrame:
        """
        Turns a 1D revenue series into a tabular feature set:
          lag_1 ... lag_n         -> previous n weeks
          roll_mean_w / roll_std_w -> rolling stats over lag_1..lag_w
          week_idx                 -> simple trend index
        Target column 'y' is the current week's revenue.
        """
        df = pd.DataFrame({"y": series})

        for lag in range(1, n_lags + 1):
            df[f"lag_{lag}"] = df["y"].shift(lag)

        for w in roll_windows:
            lag_cols = [f"lag_{i}" for i in range(1, w + 1)]
            df[f"roll_mean_{w}"] = df[lag_cols].mean(axis=1)
            df[f"roll_std_{w}"] = df[lag_cols].std(axis=1)

        df["week_idx"] = np.arange(len(df))

        df = df.dropna().reset_index(drop=True)
        feature_cols = [c for c in df.columns if c != "y"]
        return df, feature_cols


    # ==========================================================
    # HYPERPARAMETER SEARCH SPACE
    # (search is now run separately for EACH cat/state series, so every
    #  series gets its own tuned model instead of one global model -
    #  this puts XGBoost on equal footing with the per-series ARIMA orders)
    # ==========================================================

    param_dist = {
        "max_depth": [3, 4, 6, 8],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "n_estimators": [200, 400, 600, 800],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "reg_alpha": [0.0, 0.1, 0.5],
        "reg_lambda": [0.0, 0.1, 0.5],
        "min_child_weight": [1, 3, 5],
    }

    N_ITER = 15   # smaller than before since this now runs once PER series (9x total)
    CV_FOLDS = 3

    # ==========================================================
    # FAIR WALK-FORWARD SETTING
    # ARIMA refits ARIMA(history, order=best_order) at EVERY test step, so it
    # always trains on all data available up to that point. To make XGBoost an
    # apples-to-apples comparison, we now also refit XGBoost at every
    # walk-forward step on the expanding history, using the best_params found
    # once per series (the tree analogue of ARIMA's fixed best_order). This is
    # much more expensive computationally (test_size refits per series instead
    # of one), but it removes XGBoost's "static model" advantage/disadvantage.
    # ==========================================================

    REFIT_EACH_STEP = True

    # ==========================================================
    # PARAMETERS FOR PER-SERIES PLOTS
    # ==========================================================

    colors_real = {"CA": "tab:blue", "TX": "tab:green", "WI": "tab:purple"}
    colors_xgb = {"CA": "tab:orange", "TX": "tab:red", "WI": "tab:brown"}

    all_wrmsee_xgb = []
    best_params_per_series_xgb = {}
    xgb_results = []  # per-series metrics, mirrors snaive_results / ses_results / arima_results

    # ==========================================================
    # FORECASTING LOOP (tuning now happens INSIDE this loop, per series)
    # ==========================================================

    for cat in categories:
        plt.figure(figsize=(12, 6))

        print("=" * 60)
        print(f"CATEGORY : {cat}")
        print("=" * 60)

        for state in states:
            df_subset = weekly_avg_filtered[
                (weekly_avg_filtered["cat_id"] == cat) &
                (weekly_avg_filtered["state_id"] == state)
            ].sort_values("wm_yr_wk")

            series = df_subset["revenue"].values
            if len(series) < n_lags + test_size + 1:
                continue

            # ==========================================
            # TRAIN / TEST SPLIT
            # ==========================================

            train_data = series[:-test_size]
            test_data = series[-test_size:]

            train_feat_df, feature_cols = build_feature_frame(train_data, n_lags, roll_windows)
            X_train = train_feat_df[feature_cols]
            y_train = train_feat_df["y"]

            # ==========================================
            # PER-SERIES HYPERPARAMETER SEARCH
            # (done ONCE on the training data only, exactly like ARIMA's
            #  best_order search - the walk-forward refits below reuse
            #  these best_params, they don't re-search every step)
            # ==========================================

            n_splits = min(CV_FOLDS, max(2, len(X_train) // 5))

            base_model = xgb.XGBRegressor(
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1,
                verbosity=0,
            )
            search = RandomizedSearchCV(
                base_model,
                param_distributions=param_dist,
                n_iter=N_ITER,
                scoring="neg_mean_squared_error",
                cv=n_splits,
                random_state=42,
                n_jobs=-1,
                verbose=0,
            )
            search.fit(X_train, y_train)

            best_params = search.best_params_
            best_params_per_series_xgb[(cat, state)] = best_params

            print(f"{cat}-{state} | Best XGBoost params: {best_params}")

            # ==========================================
            # WALK-FORWARD PREDICTION (model refit at EVERY step,
            # exactly like the ARIMA(history, order=best_order).fit()
            # call inside the ARIMA walk-forward loop)
            # ==========================================

            history = list(train_data)
            predictions = []

            for i in range(len(test_data)):
                hist_arr = np.array(history)

                # REFIT_EACH_STEP is always True in this notebook: the model is
                # refit from scratch on the expanding history at every walk-forward
                # step, exactly like ARIMA(history, order=best_order).fit().
                hist_feat_df, _ = build_feature_frame(hist_arr, n_lags, roll_windows)
                X_hist = hist_feat_df[feature_cols]
                y_hist = hist_feat_df["y"]

                step_model = xgb.XGBRegressor(
                    **best_params,
                    objective="reg:squarederror",
                    random_state=42,
                    n_jobs=-1,
                    verbosity=0,
                )
                step_model.fit(X_hist, y_hist)

                feat_row = {}

                for lag in range(1, n_lags + 1):
                    feat_row[f"lag_{lag}"] = hist_arr[-lag]

                for w in roll_windows:
                    lag_vals = hist_arr[-w:]
                    feat_row[f"roll_mean_{w}"] = np.mean(lag_vals)
                    feat_row[f"roll_std_{w}"] = np.std(lag_vals, ddof=1)  # ddof=1 to match pandas .std() used in build_feature_frame

                feat_row["week_idx"] = len(hist_arr)

                X_input = pd.DataFrame([feat_row])[feature_cols]
                pred = step_model.predict(X_input)[0]
                predictions.append(pred)

                # walk-forward: append the actual observed value, not the prediction
                history.append(test_data[i])

            predictions = np.array(predictions)
            actual = test_data

            # ==========================================
            # METRICS
            # ==========================================

            mae = mean_absolute_error(actual, predictions)
            rmse = np.sqrt(mean_squared_error(actual, predictions))
            mape = calculate_mape(actual, predictions)
            rmsse = calculate_rmsse(train_data, actual, predictions)

            weight = revenue_weights[(cat, state)] / grand_total_revenue
            wrmsee = weight * rmsse
            all_wrmsee_xgb.append(wrmsee)


            xgb_results.append({
                "category": cat, "state": state, "model": "XGBoost",
                "MAE": mae, "RMSE": rmse, "RMSSE": rmsse,
                "WRMSSE": wrmsee, "MAPE": mape, "best_params": best_params
            })
            print(f"MAE    : {mae:.4f}")
            print(f"RMSE   : {rmse:.4f}")
            print(f"RMSSE  : {rmsse:.4f}")
            print(f"WRMSEE : {wrmsee:.4f} (Cumulative: {np.nansum(all_wrmsee_xgb):.4f})")
            print(f"MAPE   : {mape:.2f}%")
            print("-" * 50)

            # ==========================================
            # TRAIN FIT FOR PLOT
            # (a single model fit on train_data only, used purely to draw the
            #  in-sample line - same role as ARIMA's `initial_model` used for
            #  `in_sample = initial_model.fittedvalues`. It plays no part in
            #  the walk-forward predictions/metrics above.)
            # ==========================================

            initial_model = xgb.XGBRegressor(
                **best_params,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1,
                verbosity=0,
            )
            initial_model.fit(X_train, y_train)
            train_pred = initial_model.predict(X_train)

            full_pred = np.empty(len(series))
            full_pred[:] = np.nan
            full_pred[n_lags: n_lags + len(train_pred)] = train_pred
            full_pred[len(series) - len(predictions):] = predictions

            plt.plot(series[n_lags:], label=f"{state} Real", color=colors_real[state])
            plt.plot(full_pred[n_lags:], "--", label=f"{state} XGBoost", color=colors_xgb[state])

        plt.title(f"{cat} Revenue Forecast (XGBoost, per-series tuning, refit-per-step)")
        plt.xlabel("Week")
        plt.ylabel("Revenue")

        ax = plt.gca()
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, pos: f"{int(x/1000)}k" if x >= 1000 else f"{int(x)}")
        )

        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        gc.collect()

    # ==========================================================
    # OVERALL WRMSEE
    # ==========================================================

    print("\n" + "#" * 60)
    print(f"Overall WRMSEE : {np.nansum(all_wrmsee_xgb):.4f}")
    print("#" * 60)

    xgb_results_df = pd.DataFrame(xgb_results)

    print("\n" + "#" * 60)
    print(f"Overall WRMSEE (XGBoost): {xgb_results_df['WRMSSE'].sum():.4f}")
    print("#" * 60)

    xgb_results_df


### LightGBM Revenue Forecast

This section implements a **LightGBM** gradient-boosted-tree model as an alternative to the ARIMA/LSTM/CNN forecasts above.

Unlike the CNN/LSTM models, LightGBM operates on **tabular features** rather than raw sequences, so each week is converted into a row containing:

- `lag_1 ... lag_10`: revenue from the previous 10 weeks
- `roll_mean_w` / `roll_std_w`: rolling mean/std over 3, 5, and 10-week windows
- `week_idx`: a simple trend index

**Key steps:**
1. **Feature engineering** — convert each category/state revenue series into a lagged tabular dataset.
2. **Hyperparameter search** — a global `RandomizedSearchCV` over LightGBM parameters (num_leaves, depth, learning rate, etc.), analogous to the Keras Tuner step used for the CNN.
3. **Per-series fit** — refit LightGBM with the best hyperparameters for each category/state pair.
4. **Walk-forward prediction** — predict one week at a time, then append the *actual* observed test value to history before building next week's features (same strategy as the CNN/LSTM sections).
5. **Metrics** — MAE, RMSE, RMSSE, WRMSEE, and MAPE, computed with the same helper functions used elsewhere in the notebook.
6. **Visualization** — real vs. LightGBM-forecast weekly revenue per category and state.

Install once if needed: `!pip install lightgbm`

In [ ]:
!pip install lightgbm

In [ ]:
with measure_block("LightGBM"):
    import warnings
    import gc
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

    from sklearn.metrics import mean_absolute_error, mean_squared_error
    from sklearn.model_selection import RandomizedSearchCV
    import lightgbm as lgb

    warnings.filterwarnings("ignore")

    # ==========================================================
    # METRICS (identical to the ARIMA / CNN sections)
    # ==========================================================

    def calculate_rmsse(train, test, forecast):
        train = np.array(train).flatten()
        test = np.array(test).flatten()
        forecast = np.array(forecast).flatten()

        denominator = np.mean(np.diff(train) ** 2)
        if denominator < 1e-8:
            return np.nan

        numerator = np.mean((test - forecast) ** 2)
        return np.sqrt(numerator / denominator)


    def calculate_mape(y_true, y_pred):
        y_true = np.array(y_true).flatten()
        y_pred = np.array(y_pred).flatten()

        mask = y_true != 0
        if np.sum(mask) == 0:
            return np.nan

        return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


    # ==========================================================
    # FEATURE ENGINEERING
    # ==========================================================

    time_step = 10          # same lookback window used for the CNN/LSTM models
    n_lags = time_step       # number of lag features
    roll_windows = [3, 5, 10]  # rolling mean/std windows (must be <= n_lags)


    def build_feature_frame(series: np.ndarray, n_lags: int, roll_windows) -> pd.DataFrame:
        """
        Turns a 1D revenue series into a tabular feature set:
          lag_1 ... lag_n         -> previous n weeks
          roll_mean_w / roll_std_w -> rolling stats over lag_1..lag_w
          week_idx                 -> simple trend index
        Target column 'y' is the current week's revenue.
        """
        df = pd.DataFrame({"y": series})

        for lag in range(1, n_lags + 1):
            df[f"lag_{lag}"] = df["y"].shift(lag)

        for w in roll_windows:
            lag_cols = [f"lag_{i}" for i in range(1, w + 1)]
            df[f"roll_mean_{w}"] = df[lag_cols].mean(axis=1)
            df[f"roll_std_{w}"] = df[lag_cols].std(axis=1)

        df["week_idx"] = np.arange(len(df))

        df = df.dropna().reset_index(drop=True)
        feature_cols = [c for c in df.columns if c != "y"]
        return df, feature_cols


    # ==========================================================
    # HYPERPARAMETER SEARCH SPACE
    # (search is now run separately for EACH cat/state series, so every
    #  series gets its own tuned model instead of one global model -
    #  this puts LightGBM on equal footing with the per-series ARIMA orders)
    # ==========================================================

    param_dist = {
        "num_leaves": [15, 31, 63, 127],
        "max_depth": [-1, 4, 6, 8],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "n_estimators": [200, 400, 600, 800],
        "min_child_samples": [5, 10, 20, 30],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "reg_alpha": [0.0, 0.1, 0.5],
        "reg_lambda": [0.0, 0.1, 0.5],
    }

    N_ITER = 15   # smaller than before since this now runs once PER series (9x total)
    CV_FOLDS = 3

    # ==========================================================
    # FAIR WALK-FORWARD SETTING
    # ARIMA refits ARIMA(history, order=best_order) at EVERY test step, so it
    # always trains on all data available up to that point. To make LightGBM an
    # apples-to-apples comparison, we now also refit LightGBM at every
    # walk-forward step on the expanding history, using the best_params found
    # once per series (the tree analogue of ARIMA's fixed best_order). This is
    # much more expensive computationally (test_size refits per series instead
    # of one), but it removes LightGBM's "static model" advantage/disadvantage.
    # ==========================================================

    REFIT_EACH_STEP = True

    # ==========================================================
    # PARAMETERS FOR PER-SERIES PLOTS
    # ==========================================================

    colors_real = {"CA": "tab:blue", "TX": "tab:green", "WI": "tab:purple"}
    colors_lgbm = {"CA": "tab:orange", "TX": "tab:red", "WI": "tab:brown"}

    all_wrmsee_lgbm = []
    best_params_per_series_lgbm = {}
    lgbm_results = []  # per-series metrics, mirrors snaive_results / ses_results / arima_results

    # ==========================================================
    # FORECASTING LOOP (tuning now happens INSIDE this loop, per series)
    # ==========================================================

    for cat in categories:
        plt.figure(figsize=(12, 6))

        print("=" * 60)
        print(f"CATEGORY : {cat}")
        print("=" * 60)

        for state in states:
            df_subset = weekly_avg_filtered[
                (weekly_avg_filtered["cat_id"] == cat) &
                (weekly_avg_filtered["state_id"] == state)
            ].sort_values("wm_yr_wk")

            series = df_subset["revenue"].values
            if len(series) < n_lags + test_size + 1:
                continue

            # ==========================================
            # TRAIN / TEST SPLIT
            # ==========================================

            train_data = series[:-test_size]
            test_data = series[-test_size:]

            train_feat_df, feature_cols = build_feature_frame(train_data, n_lags, roll_windows)
            X_train = train_feat_df[feature_cols]
            y_train = train_feat_df["y"]

            # ==========================================
            # PER-SERIES HYPERPARAMETER SEARCH
            # (done ONCE on the training data only, exactly like ARIMA's
            #  best_order search - the walk-forward refits below reuse
            #  these best_params, they don't re-search every step)
            # ==========================================

            n_splits = min(CV_FOLDS, max(2, len(X_train) // 5))

            base_model = lgb.LGBMRegressor(objective="regression", random_state=42, verbosity=-1)
            search = RandomizedSearchCV(
                base_model,
                param_distributions=param_dist,
                n_iter=N_ITER,
                scoring="neg_mean_squared_error",
                cv=n_splits,
                random_state=42,
                n_jobs=-1,
                verbose=0,
            )
            search.fit(X_train, y_train)

            best_params = search.best_params_
            best_params_per_series_lgbm[(cat, state)] = best_params

            print(f"{cat}-{state} | Best LightGBM params: {best_params}")

            # ==========================================
            # WALK-FORWARD PREDICTION (model refit at EVERY step,
            # exactly like the ARIMA(history, order=best_order).fit()
            # call inside the ARIMA walk-forward loop)
            # ==========================================

            history = list(train_data)
            predictions = []

            for i in range(len(test_data)):
                hist_arr = np.array(history)

                # REFIT_EACH_STEP is always True in this notebook: the model is
                # refit from scratch on the expanding history at every walk-forward
                # step, exactly like ARIMA(history, order=best_order).fit().
                hist_feat_df, _ = build_feature_frame(hist_arr, n_lags, roll_windows)
                X_hist = hist_feat_df[feature_cols]
                y_hist = hist_feat_df["y"]

                step_model = lgb.LGBMRegressor(
                    **best_params, objective="regression", random_state=42, verbosity=-1
                )
                step_model.fit(X_hist, y_hist)

                feat_row = {}

                for lag in range(1, n_lags + 1):
                    feat_row[f"lag_{lag}"] = hist_arr[-lag]

                for w in roll_windows:
                    lag_vals = hist_arr[-w:]
                    feat_row[f"roll_mean_{w}"] = np.mean(lag_vals)
                    feat_row[f"roll_std_{w}"] = np.std(lag_vals, ddof=1)  # ddof=1 to match pandas .std() used in build_feature_frame

                feat_row["week_idx"] = len(hist_arr)

                X_input = pd.DataFrame([feat_row])[feature_cols]
                pred = step_model.predict(X_input)[0]
                predictions.append(pred)

                # walk-forward: append the actual observed value, not the prediction
                history.append(test_data[i])

            predictions = np.array(predictions)
            actual = test_data

            # ==========================================
            # METRICS
            # ==========================================

            mae = mean_absolute_error(actual, predictions)
            rmse = np.sqrt(mean_squared_error(actual, predictions))
            mape = calculate_mape(actual, predictions)
            rmsse = calculate_rmsse(train_data, actual, predictions)

            weight = revenue_weights[(cat, state)] / grand_total_revenue
            wrmsee = weight * rmsse
            all_wrmsee_lgbm.append(wrmsee)


            lgbm_results.append({
                "category": cat, "state": state, "model": "LightGBM",
                "MAE": mae, "RMSE": rmse, "RMSSE": rmsse,
                "WRMSSE": wrmsee, "MAPE": mape, "best_params": best_params
            })
            print(f"MAE    : {mae:.4f}")
            print(f"RMSE   : {rmse:.4f}")
            print(f"RMSSE  : {rmsse:.4f}")
            print(f"WRMSEE : {wrmsee:.4f} (Cumulative: {np.nansum(all_wrmsee_lgbm):.4f})")
            print(f"MAPE   : {mape:.2f}%")
            print("-" * 50)

            # ==========================================
            # TRAIN FIT FOR PLOT
            # (a single model fit on train_data only, used purely to draw the
            #  in-sample line - same role as ARIMA's `initial_model` used for
            #  `in_sample = initial_model.fittedvalues`. It plays no part in
            #  the walk-forward predictions/metrics above.)
            # ==========================================

            initial_model = lgb.LGBMRegressor(
                **best_params, objective="regression", random_state=42, verbosity=-1
            )
            initial_model.fit(X_train, y_train)
            train_pred = initial_model.predict(X_train)

            full_pred = np.empty(len(series))
            full_pred[:] = np.nan
            full_pred[n_lags: n_lags + len(train_pred)] = train_pred
            full_pred[len(series) - len(predictions):] = predictions

            plt.plot(series[n_lags:], label=f"{state} Real", color=colors_real[state])
            plt.plot(full_pred[n_lags:], "--", label=f"{state} LightGBM", color=colors_lgbm[state])

        plt.title(f"{cat} Revenue Forecast (LightGBM, per-series tuning, refit-per-step)")
        plt.xlabel("Week")
        plt.ylabel("Revenue")

        ax = plt.gca()
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, pos: f"{int(x/1000)}k" if x >= 1000 else f"{int(x)}")
        )

        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        gc.collect()

    # ==========================================================
    # OVERALL WRMSEE
    # ==========================================================

    print("\n" + "#" * 60)
    print(f"Overall WRMSEE : {np.nansum(all_wrmsee_lgbm):.4f}")
    print("#" * 60)

    lgbm_results_df = pd.DataFrame(lgbm_results)

    print("\n" + "#" * 60)
    print(f"Overall WRMSEE (LightGBM): {lgbm_results_df['WRMSSE'].sum():.4f}")
    print("#" * 60)

    lgbm_results_df


### Model Comparison: Accuracy, Computation Time, Memory Usage, Complexity

This section aggregates the `measure_block()` timing/memory captured in every model section above with each model's own `*_results_df` (WRMSSE/MAPE) and a fixed hyperparameter-count-based complexity rubric, into one 0-100 comparison table and chart.

**Run every model section above first** (Seasonal Naive → SES → ARIMA → Random Forest → XGBoost → LightGBM), in order, before running this cell - it needs `model_perf` and every `*_results_df` to already exist.

In [ ]:
# ==========================================================
# FINAL MODEL COMPARISON: ACCURACY vs TIME vs MEMORY vs COMPLEXITY
# ==========================================================
# Pulls together:
#   1) ACCURACY  - from each model's own *_results_df (WRMSSE = the
#                  M5-style weighted RMSSE used everywhere in this
#                  notebook; lower is better) + average MAPE
#   2) TIME      - wall-clock seconds captured by measure_block() for
#                  the WHOLE section (hyperparameter search + every
#                  walk-forward refit across all 9 category/state series)
#   3) MEMORY    - peak Python-object memory (MB) captured by
#                  tracemalloc inside measure_block() for that same
#                  section
#   4) COMPLEXITY- NOT measured at runtime. It is a fixed rubric based
#                  on (a) how many hyperparameters are tuned by the
#                  RandomizedSearchCV / grid-search in that section and
#                  (b) the underlying algorithm family (closed-form /
#                  single-parameter smoothing < iterative ML estimation
#                  < single decision tree < tree ensembles / boosting).
#                  Edit COMPLEXITY_RUBRIC below if you'd rather score
#                  complexity differently (e.g. by counting model
#                  parameters/trees instead of tuned hyperparameters).
#
# All four scores are also rescaled to a common 0-100 "higher is
# better" scale so they can be compared/plotted side by side:
#   accuracy_score   = 100 * (1 - normalized WRMSSE)   -> higher = more accurate
#   time_score       = 100 * (1 - normalized time_sec) -> higher = faster
#   memory_score     = 100 * (1 - normalized peak_mem)  -> higher = lighter
#   complexity_score = 100 * (1 - normalized raw complexity) -> higher = simpler
# ==========================================================

import numpy as np
import pandas as pd

# ---------- 1) ACCURACY: aggregate each model's results_df ----------
all_results_df = pd.concat(
    [snaive_results_df, ses_results_df, arima_results_df,
     rf_results_df, xgb_results_df, lgbm_results_df],
    ignore_index=True
)

accuracy_summary = all_results_df.groupby("model").agg(
    WRMSSE=("WRMSSE", "sum"),      # sum of weighted per-series RMSSE = overall WRMSSE, same as the
                                    # "Overall WRMSSE" printed at the end of each section
    RMSSE_mean=("RMSSE", "mean"),
    MAE_mean=("MAE", "mean"),
    RMSE_mean=("RMSE", "mean"),
    MAPE_mean=("MAPE", "mean"),
)

# ---------- 2) & 3) TIME + MEMORY: from model_perf (measure_block) ----------
perf_df = pd.DataFrame(model_perf).T
perf_df.index.name = "model"

# ---------- 4) COMPLEXITY: fixed rubric, not runtime-measured ----------
COMPLEXITY_RUBRIC = {
    # model_name: (n_hyperparams_tuned, raw_complexity_1_to_10, note)
    "Seasonal Naive": (1, 1, "1 hyperparameter (season_length), 4 candidates, O(1) lookup - no model fitting at all"),
    "SES":            (1, 2, "1 hyperparameter (alpha), refit via MLE every walk-forward step, closed-form-ish optimization"),
    "ARIMA":          (3, 5, "3 hyperparameters (p,d,q), 18-combo AIC grid search once + non-convex MLE refit every step"),
    "Random Forest":  (5, 7, "5 hyperparameters tuned (45 CV fits via RandomizedSearchCV) + ensemble of up to 800 trees refit every step"),
    "XGBoost":        (8, 8, "8 hyperparameters tuned (45 CV fits) + boosted ensemble up to 800 trees w/ regularization, refit every step"),
    "LightGBM":       (9, 8, "9 hyperparameters tuned (45 CV fits) + leaf-wise boosted ensemble up to 800 trees w/ regularization, refit every step"),
}
complexity_df = pd.DataFrame(COMPLEXITY_RUBRIC).T
complexity_df.columns = ["n_hyperparams_tuned", "raw_complexity_1_10", "complexity_note"]
complexity_df.index.name = "model"

# ---------- combine everything ----------
comparison = accuracy_summary.join(perf_df).join(complexity_df[["n_hyperparams_tuned", "raw_complexity_1_10", "complexity_note"]])


def normalize_lower_is_better(series):
    """0-100 score where the smallest raw value -> 100, largest raw value -> 0."""
    s = series.astype(float)
    if s.max() == s.min():
        return pd.Series(100.0, index=s.index)
    return 100 * (1 - (s - s.min()) / (s.max() - s.min()))


comparison["accuracy_score"] = normalize_lower_is_better(comparison["WRMSSE"])
comparison["time_score"] = normalize_lower_is_better(comparison["time_sec"])
comparison["memory_score"] = normalize_lower_is_better(comparison["peak_mem_MB"])
comparison["complexity_score"] = normalize_lower_is_better(comparison["raw_complexity_1_10"])

comparison["overall_score"] = comparison[
    ["accuracy_score", "time_score", "memory_score", "complexity_score"]
].mean(axis=1)

comparison = comparison.sort_values("overall_score", ascending=False)

print("=" * 70)
print(" MODEL COMPARISON - ACCURACY / TIME / MEMORY / COMPLEXITY")
print("=" * 70)
display_cols = [
    "WRMSSE", "MAPE_mean", "accuracy_score",
    "time_sec", "time_score",
    "peak_mem_MB", "memory_score",
    "raw_complexity_1_10", "complexity_score",
    "overall_score",
]
print(comparison[display_cols].round(2))

print("\nNote: complexity_score is rubric-based (see COMPLEXITY_RUBRIC above), "
      "not something tracemalloc/time can measure directly.")

# ---------- visual comparison ----------
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

comparison["accuracy_score"].plot(kind="bar", ax=axes[0, 0], color="tab:blue", title="Accuracy score (higher = lower WRMSSE)")
comparison["time_score"].plot(kind="bar", ax=axes[0, 1], color="tab:orange", title="Time score (higher = faster)")
comparison["memory_score"].plot(kind="bar", ax=axes[1, 0], color="tab:green", title="Memory score (higher = lighter)")
comparison["complexity_score"].plot(kind="bar", ax=axes[1, 1], color="tab:red", title="Complexity score (higher = simpler)")

for ax in axes.flat:
    ax.set_ylim(0, 100)
    ax.set_ylabel("score (0-100)")
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, axis='y', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

comparison[display_cols]
